# Быстрый эксперимент

Черновик для проверки идей в одинаковых условиях на всех трёх фолдах: запуск - здесь (Run All) или на Kaggle, `make run NB=05` из папки проекта.

Сейчас - поверх четвёртого сабмита (целое количество под SMAPE, порог 0.5): **ансамбль**, как у победителя M5 - среднее нескольких моделей надёжнее одной. Вероятности классификаторов усредняются, количества регрессоров - в логарифмах (`ensemble` в методе `pipeline`). Участники: модель сабмита, она же с другим сидом и она же с глубиной 8 (для разнообразия). Каждая модель учится один раз, сочетания считаются поверх кэша.

Сравниваем с четвёртым сабмитом, посчитанным в этом же запуске (на GPU результат от запуска к запуску чуть плавает). Вариант берём, только если он не хуже на обоих сезонных фолдах, а на `recent` хуже не больше чем на 0.005.

In [1]:
import sys; sys.path[:0] = ["../src", "../../src"]   # код проекта - в src/ (ноутбук в notebooks/ или notebooks/05_experiments/)

import pandas as pd
import core
from core import *
import classifier, regressor, pipeline

# быстрее, чем в 03/04: на Kaggle - каждая 2-я неделя отсчёта (по кривой из 03 этого хватает); деревьев - по 400, как в модулях
if ON_KAGGLE:
    classifier.ORIGIN_STEP = regressor.ORIGIN_STEP = 2
sales, stock = load_data()
classifier.PARAMS, regressor.PARAMS, classifier.HALF_LIFE, classifier.ORIGIN_STEP

({'iterations': 400,
  'learning_rate': 0.1,
  'depth': 6,
  'loss_function': 'Logloss',
  'max_ctr_complexity': 1,
  'random_seed': 42,
  'verbose': 0,
  'allow_writing_files': False,
  'thread_count': -1,
  'one_hot_max_size': 64,
  'task_type': 'GPU'},
 {'iterations': 400,
  'learning_rate': 0.1,
  'depth': 6,
  'loss_function': 'RMSE',
  'max_ctr_complexity': 1,
  'random_seed': 42,
  'verbose': 0,
  'allow_writing_files': False,
  'thread_count': -1,
  'task_type': 'GPU'},
 26,
 2)

## Варианты

`VARIANTS` - параметры метода `pipeline`; первый - четвёртый сабмит, с ним и сравниваем.

In [2]:
SUB = {"threshold": 0.5, "integer": "smape"}                    # четвёртый сабмит
SEED7, DEPTH8 = {"random_seed": 7}, {"depth": 8, "learning_rate": 0.05}
VARIANTS = {
    "сабмит: одна модель": SUB,
    "ансамбль: + сид 7": {**SUB, "ensemble": [{}, SEED7]},
    "ансамбль: + глубина 8": {**SUB, "ensemble": [{}, DEPTH8]},
    "ансамбль: все три": {**SUB, "ensemble": [{}, SEED7, DEPTH8]},
    "ансамбль: все три, порог 0.55": {**SUB, "threshold": 0.55, "ensemble": [{}, SEED7, DEPTH8]},
}
SEASON = ["season2025", "season2024"]

## Сетка на трёх фолдах

Модели учатся один раз на фолд, варианты перебираются поверх них.

In [3]:
rows = []
for fold, cut in FOLDS.items():
    pairs, s_cut, st_cut, truth, base = core._fold_data(fold, cut)
    for name, params in VARIANTS.items():
        pred = make_forecast(s_cut, st_cut, "pipeline", pairs=pairs, **params)
        m = compute_metrics(truth, pred)
        rows.append({"вариант": name, "фолд": fold, **m, "score": score_minmax(m, base),
                     "прогноз / факт": pred.quantity.sum() / truth.quantity.sum()})
    print(fold, "готов", flush=True)
grid = pd.DataFrame(rows)
grid.pivot_table(index="вариант", columns="фолд", values="smape")[list(FOLDS)].round(2)

season2025 готов


season2024 готов


recent готов


фолд,season2025,season2024,recent
вариант,,,
ансамбль: + глубина 8,57.55,42.89,58.74
ансамбль: + сид 7,57.53,42.75,58.78
ансамбль: все три,57.57,42.79,58.74
"ансамбль: все три, порог 0.55",57.66,42.83,58.22
сабмит: одна модель,57.58,43.01,58.72


## Выбор

Проходит вариант, который не хуже четвёртого сабмита на обоих сезонных фолдах и на `recent` хуже не больше чем на 0.005; из прошедших - с лучшим средним по двум сезонным фолдам.

In [4]:
scores = grid.pivot_table(index="вариант", columns="фолд", values="score")[list(FOLDS)]
BASE = scores.loc["сабмит: одна модель"]
scores["по сезону"] = scores[SEASON].mean(axis=1)
scores["проходит"] = (scores[SEASON] <= BASE[SEASON]).all(axis=1) & (scores["recent"] <= BASE["recent"] + 0.005)
scores.loc["сабмит: одна модель", "проходит"] = False                          # сам с собой не сравниваем
BEST = scores[scores["проходит"]]["по сезону"].idxmin() if scores["проходит"].any() else None
print("четвёртый сабмит:", BASE[list(FOLDS)].round(4).to_dict(), "| по сезону", round(BASE[SEASON].mean(), 4))
print("лучший вариант:", BEST, scores.loc[BEST, list(FOLDS) + ["по сезону"]].astype(float).round(4).to_dict() if BEST else "- ни один не прошёл, сабмит не меняем")
scores.sort_values("по сезону").round(4)

четвёртый сабмит: {'season2025': 0.811, 'season2024': 0.8251, 'recent': 0.7878} | по сезону 0.8181


TypeError: Expected numeric dtype, got object instead.

## Метрики и журнал

Лучший вариант - `backtest` на трёх фолдах (модели из кэша), запись в журнале - с пометкой "эксперимент".

In [5]:
if BEST:
    display(backtest("pipeline", note=f"эксперимент: {BEST}", **VARIANTS[BEST]))

,run,method,params,fold,smape,wape,rmse,bias,rmsle,score,pred/true,note
0,1,pipeline,"{'threshold': 0.5, 'integer': 'smape', 'ensemb...",season2025,57.531,51.201,2.645,81.060,0.506,0.809,0.785,эксперимент: ансамбль: + сид 7
1,1,pipeline,"{'threshold': 0.5, 'integer': 'smape', 'ensemb...",season2024,42.753,62.845,1.927,86.626,0.434,0.823,0.732,эксперимент: ансамбль: + сид 7
2,1,pipeline,"{'threshold': 0.5, 'integer': 'smape', 'ensemb...",recent,58.783,47.111,3.348,76.596,0.510,0.788,0.840,эксперимент: ансамбль: + сид 7


## Сабмит

Лучший вариант - на всей истории по 2026-07-26 (модели учатся заново), файл - `SUBMISSION_CSV` (на Kaggle - в выводах запуска, `make pull NB=05` забирает его в `outputs/kaggle/05_experiment/`). Не прошёл ни один - сабмит прежний.

In [6]:
if BEST:
    sub = make_forecast(sales, stock, "pipeline", **VARIANTS[BEST])
    print(check_submission(sub) or "формат ок")
    sub.to_csv(SUBMISSION_CSV, index=False, date_format="%Y-%m-%d")
    print(len(sub), "строк, недели", sub.week.min().date(), "...", sub.week.max().date(),
          "| штук за 9 недель:", round(sub.quantity.sum()), "| вариант:", BEST, "->", SUBMISSION_CSV)
else:
    print("сабмит не меняем")

формат ок


79101 строк, недели 2026-07-27 ... 2026-09-21 | штук за 9 недель: 221311 | вариант: ансамбль: + сид 7 -> /kaggle/working/submission.csv
